# Test Technique : Wiremind

**Objectif :** Construire un modèle prédictif robuste pour estimer la demande non contrainte des trains. 

Dans un contexte de Revenue Management, la prévision de la demande est le pilier central de l'optimisation tarifaire. Avant de modéliser, cette analyse exploratoire vise à valider la cohérence des données et à identifier les dynamiques d'achat (Booking Curve, élasticité-prix, saisonnalité, cannibalisation) afin de guider notre stratégie de Feature Engineering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import timedelta

# Configuration globale pour la lisibilité des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Chargement des données
train = pd.read_parquet('./data/cayzn_train.parquet')
test = pd.read_parquet('./data/cayzn_test.parquet')

print(f"Dimensions du dataset d'entraînement : {train.shape}")

## 1. Analyse Exploratoire des Données 
### 1.1. La Variable Cible et la Booking Curve

In [ ]:
plt.figure(figsize=(10, 6))
booking_curve = train.groupby('sale_day_x')['demand'].mean().reset_index()
sns.lineplot(data=booking_curve, x='sale_day_x', y='demand', marker="o")
plt.title("Booking Curve Moyenne : Demande vs Jours avant départ")
plt.xlabel("Jours avant le départ (sale_day_x)")
plt.ylabel("Demande Moyenne")
plt.gca().invert_xaxis() 
plt.show()

La relation est fortement non-linéaire. La demande moyenne est quasi-nulle jusqu'à J-40, puis grimpe de manière exponentielle jusqu'à atteindre un pic de plus de 16 billets en moyenne le jour J. Les modèles telles que LightGBM/CatBoost seront beaucoup plus adaptés qu'une régression linéaire basique pour capter cette accélération.

### 1.2. Dynamique Tarifaire et Élasticité-Prix

In [ ]:
plt.figure(figsize=(14, 6))
price_stats = train.groupby('sale_day_x')['price'].agg(['mean', 'std']).reset_index()

sns.lineplot(data=price_stats, x='sale_day_x', y='mean', color='red', marker="o", label='Prix Moyen')
plt.fill_between(price_stats['sale_day_x'],
                 price_stats['mean'] - price_stats['std'],
                 price_stats['mean'] + price_stats['std'],
                 color='red', alpha=0.2, label='Écart Type (+/- 1 std)')

plt.title("Évolution et Dispersion du Prix à l'approche du départ")
plt.xlabel("Jours avant le départ (sale_day_x)")
plt.ylabel("Prix (EUR)")
plt.gca().invert_xaxis()
plt.legend()
plt.show()

Le prix moyen monte de 22€ (J-90) à un pic de 28€ vers J-10, puis rechute étonnamment à 24€ en dernière minute. Surtout, l'écart-type (zone rouge) explose à l'approche du départ, avec des prix allant de moins de 10€ à plus de 50€. Le système polarise son offre en dernière minute : brader les trains vides, facturer au prix fort les trains pleins.

### 1.3. Saisonnalité et Impacts Exogènes

In [ ]:
# Saisonnalité macroscopique (Jours fériés et Week-ends)
train['departure_date'] = pd.to_datetime(train['departure_date'])

daily_seasonality = train.groupby('departure_date').agg(
    total_demand=('demand', 'sum'),
    holiday_origin=('origin_current_public_holiday', 'max'),
    holiday_dest=('destination_current_public_holiday', 'max'),
    weekday=('od_origin_weekday', 'max')
).reset_index()

daily_seasonality['is_holiday'] = (daily_seasonality['holiday_origin'] == 1) | (daily_seasonality['holiday_dest'] == 1)
daily_seasonality['is_weekend'] = daily_seasonality['weekday'].isin([5, 6])

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily_seasonality['departure_date'], daily_seasonality['total_demand'], color='#1f77b4', linewidth=2, label='Demande Totale')

added_holiday, added_weekend = False, False
for _, row in daily_seasonality.iterrows():
    start_date = row['departure_date']
    end_date = start_date + pd.Timedelta(days=1)
    if row['is_holiday']:
        ax.axvspan(start_date, end_date, color='red', alpha=0.3, label='Jour Férié' if not added_holiday else "")
        added_holiday = True
    elif row['is_weekend']:
        ax.axvspan(start_date, end_date, color='gray', alpha=0.2, label='Week-end' if not added_weekend else "")
        added_weekend = True

ax.set_title("Saisonnalité de la Demande Globale en fonction du Jour de l'Année")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

Le graphique affiche des pics massifs parfaitement alignés avec les jours fériés (bandes rouges). Plus inquiétant pour notre modèle d'apprentissage : un effondrement quasi total de la demande au printemps 2020. C'est l'impact direct du confinement lié au COVID-19. On remarque également une baisse de la demande liée aux grèves fin 2019 Début 2020. La gestion de ces périodes aberrantes est primordiale. 

### 1.4. Audit de la Qualité des Données (Détection de Gaps)

In [ ]:
def detect_data_gaps(df, date_column='departure_date'):
    existing_dates = df[date_column].dt.date.dropna().unique()
    if len(existing_dates) == 0:
        return
        
    full_calendar = pd.date_range(start=existing_dates.min(), end=existing_dates.max()).date
    missing_dates = sorted(list(set(full_calendar) - set(existing_dates)))
    
    gaps, start_gap, prev_date = [], missing_dates[0], missing_dates[0]
    for current_date in missing_dates[1:]:
        if (current_date - prev_date).days > 1:
            gaps.append((start_gap, prev_date))
            start_gap = current_date
        prev_date = current_date
    gaps.append((start_gap, prev_date))
    
    print("--- PÉRIODES MANQUANTES DÉTECTÉES ---")
    for start, end in gaps:
        print(f"Du {start} au {end} : {(end - start).days + 1} jours manquants")

detect_data_gaps(train)

### 1.5. Feature Engineering
Notre stratégie s'appuie sur la création de features métier pour contextualiser l'offre sans recourir à des données externes.

In [ ]:
def feature_engineering_pipeline(df):
    print("Démarrage du Feature Engineering...")
    df_fe = df.copy()
    df_fe['departure_date'] = pd.to_datetime(df_fe['departure_date'])
    
    # 1. FLAG ANOMALIES (Basé sur l'audit de qualité)
    conditions = [
        df_fe['departure_date'].between('2019-10-03', '2019-10-04'),
        df_fe['departure_date'].between('2019-12-01', '2020-01-31'), # Grèves
        df_fe['departure_date'].between('2020-03-10', '2020-03-11'),
        df_fe['departure_date'].between('2020-03-15', '2020-06-25')  # Confinement
    ]
    df_fe['is_off_service'] = np.where(np.logical_or.reduce(conditions), 0, 1)

    # 2. Tri chronologique et création d'un ID unique pour chaque train
    if 'Unique_Train_ID' not in df_fe.columns:
        df_fe['Unique_Train_ID'] = df_fe['departure_date'].astype(str) + "_" + \
                                   df_fe['od_origin_time'].astype(str) + "_" + \
                                   df_fe['origin_station_name'] + "_" + \
                                   df_fe['destination_station_name']
                                   
    df_fe = df_fe.sort_values(by=['Unique_Train_ID', 'sale_day_x']).reset_index(drop=True)

    # 3. COMPORTEMENT D'ACHAT
    df_fe['is_last_minute'] = np.where(df_fe['sale_day_x'] >= -3, 1, 0)
    df_fe['is_early_bird'] = np.where(df_fe['sale_day_x'] <= -60, 1, 0)

    # 4. CONTEXTUALISATION DU PRIX 
    df_fe['od_weekday_mean_price'] = df_fe.groupby([
        'origin_station_name', 'destination_station_name', 'od_origin_weekday'
    ])['price'].transform('mean')

    df_fe['price_ratio_to_od_weekday_mean'] = df_fe['price'] / (df_fe['od_weekday_mean_price'] + 0.001)
    df_fe['max_price_seen_so_far'] = df_fe.groupby('Unique_Train_ID')['price'].cummax()
    df_fe['min_price_seen_so_far'] = df_fe.groupby('Unique_Train_ID')['price'].cummin()

    # 5. PUISSANCE DE LIGNE
    df_fe['daily_od_frequency'] = df_fe.groupby([
        'departure_date', 'origin_station_name', 'destination_station_name'
    ])['Unique_Train_ID'].transform('nunique')

    df_fe['demand_rolling_avg_7d'] = df_fe.groupby('Unique_Train_ID')['demand'] \
                                          .shift(1) \
                                          .rolling(window=7, min_periods=1) \
                                          .mean() \
                                          .fillna(0)

    print(f"Feature Engineering terminé. Dimensions : {df_fe.shape}")
    return df_fe

train_fe = feature_engineering_pipeline(train)

## 2. Stratégie de Validation et Modélisation
Pour ce problème de Revenue Management, CatBoost s'impose comme l'algorithme idéal grâce à sa gestion native du *Target Encoding* sur les nombreuses variables catégorielles comme pour les gares ici.

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

target = 'demand'
cat_features = ['origin_station_name', 'destination_station_name']
features = [col for col in train_fe.columns if col not in ['demand', 'departure_date', 'sale_date', 'Unique_Train_ID']]

for col in cat_features:
    train_fe[col] = train_fe[col].astype(str)

# Séparation Temporelle (Out-of-Time Split)
train_fe = train_fe.sort_values('departure_date')
split_index = int(len(train_fe) * 0.85)
split_date = train_fe.iloc[split_index]['departure_date']

X_train, y_train = train_fe[train_fe['departure_date'] < split_date][features], train_fe[train_fe['departure_date'] < split_date][target]
X_val, y_val = train_fe[train_fe['departure_date'] >= split_date][features], train_fe[train_fe['departure_date'] >= split_date][target]

model_cb = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    loss_function='MAE',
    eval_metric='MAE',
    cat_features=cat_features,
    random_seed=42,
    verbose=100
)

model_cb.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, use_best_model=True)

L'entraînement montre de bons résultats. L'Early Stopping s'est déclenché à la 314ème itération, nous évitant l'overfitting. Les métriques de validation (MAE: 1.15 billets) sont bonnes pour un problème aussi volatil que la prévision de dernière minute.

## 3. Model Validation & Analyse Business (Test Set)
En Revenue Management, l'objectif est d'optimiser le Revenu. Évaluer une MAE de volume ne suffit pas ; nous devons évaluer l'impact financier réel de l'erreur du modèle.

In [ ]:
test_fe = feature_engineering_pipeline(test)
for col in cat_features:
    test_fe[col] = test_fe[col].astype(str)

X_test, y_test = test_fe[features], test_fe[target]
y_pred_test = np.maximum(0, model_cb.predict(X_test))

test_fe['predicted_demand'] = y_pred_test
test_fe['demand_error'] = test_fe['predicted_demand'] - test_fe['demand']
test_fe['demand_abs_error'] = test_fe['demand_error'].abs()

test_fe['true_revenue'] = test_fe['demand'] * test_fe['price']
test_fe['predicted_revenue'] = test_fe['predicted_demand'] * test_fe['price']
test_fe['revenue_abs_error'] = (test_fe['predicted_revenue'] - test_fe['true_revenue']).abs()

print("--- PERFORMANCES GLOBALES SUR LE SET DE TEST ---")
print(f"MAE Demande (Volume) : {test_fe['demand_abs_error'].mean():.2f} billets par prédiction")
print(f"MAE Revenu (Impact Financier) : {test_fe['revenue_abs_error'].mean():.2f} € par prédiction")

### 3.1 Évolution de l'erreur dans le temps

In [ ]:
plt.figure(figsize=(14, 6))
error_by_day_x = test_fe.groupby('sale_day_x').agg(
    mean_demand_error=('demand_abs_error', 'mean'),
    mean_revenue_error=('revenue_abs_error', 'mean')
).reset_index()

ax1 = sns.lineplot(data=error_by_day_x, x='sale_day_x', y='mean_demand_error', color='blue', label='Erreur Absolue Demande')
ax1.set_ylabel('Erreur Demande (Billets)', color='blue')
ax2 = ax1.twinx()
sns.lineplot(data=error_by_day_x, x='sale_day_x', y='mean_revenue_error', color='green', label='Erreur Absolue Revenu', ax=ax2)
ax2.set_ylabel('Erreur Revenu (€)', color='green')
plt.title("Évolution de l'erreur de prédiction selon l'horizon de vente")
ax1.invert_xaxis()
plt.show()

Les deux courbes (Demande en bleu et Revenu en vert) montrent une corrélation presque parfaite. L'erreur financière grimpe en flèche à l'approche du départ (J-10 à J-0) pour atteindre un risque de plus de 175€ par prédiction le jour même. Cela confirme que l'enjeu financier de l'algorithme se joue intégralement dans les 10 derniers jours.

### 3.2 Les Trajets critiques (OD Level)

In [ ]:
test_fe['OD'] = test_fe['origin_station_name'] + " - " + test_fe['destination_station_name']
error_by_od = test_fe.groupby('OD').agg(
    mean_revenue_error=('revenue_abs_error', 'mean'),
    prediction_count=('demand', 'count')
).sort_values(by='mean_revenue_error', ascending=False)

worst_ods = error_by_od[error_by_od['prediction_count'] > 50].head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=worst_ods['mean_revenue_error'], y=worst_ods.index, palette='Reds_r')
plt.title("Top 10 des Trajets (OD) avec la plus forte erreur financière moyenne")
plt.xlabel("Erreur Absolue Moyenne sur le Revenu (€)")
plt.show()

Le trajet `rb - bb` est de loin le plus problématique, avec une erreur de revenu excédant 90€ par prédiction. Le flux `cpe - ag` dépasse également les 80€. Ces lignes ont très probablement un comportement atypique, une forte composante événementielle locale, ou sont particulièrement sensibles à une concurrence par d'autres moyens de transport.

### 3.3 Biais Systématique et Sensibilité Tarifaire

In [ ]:
test_fe['price_tier'] = pd.qcut(test_fe['price'], q=4, labels=['1. Low Price', '2. Medium-Low', '3. Medium-High', '4. Premium Price'])
error_by_price = test_fe.groupby('price_tier').agg(
    bias_demand=('demand_error', 'mean'),
    mae_revenue=('revenue_abs_error', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=error_by_price, x='price_tier', y='bias_demand', ax=axes[0], palette='coolwarm')
axes[0].set_title("Biais de Prédiction par Niveau de Prix")
axes[0].set_ylabel("Biais Moyen (Billets)")
axes[0].axhline(0, color='black', linestyle='--')

sns.barplot(data=error_by_price, x='price_tier', y='mae_revenue', ax=axes[1], palette='Oranges')
axes[1].set_title("Impact Financier de l'erreur par Niveau de Prix")
axes[1].set_ylabel("Erreur Absolue Moyenne du Revenu (€)")
plt.show()

On déduit deux tendances principales:
- Biais Négatif Global: Le graphique de gauche prouve que le modèle sous-estime systématiquement la demande, atteignant un pic de -0.62 billets pour le segment "Medium-Low".
- Le danger du Premium: Bien que le biais en volume soit limité pour les billets "Premium" (Tier 4), l'erreur financière y explose (plus de 110€ en moyenne). L'algorithme actuel a des difficultés à anticiper avec précision ces achats de dernière minute peu sensibles au prix (typiquement la clientèle B2B), ce qui peut induire une mauvaise fermeture des classes tarifaires.

## 6. Conclusion et Pistes d'Amélioration

Ce test technique met en évidence un pipeline prédictif performant, exploitant avec succès les mécaniques propres au Revenue Management (Momentum des ventes, relativité tarifaire). 

**Principaux conclusions :**
* CatBoost gère correctment la topologie du dataset avec une MAE globale de 1.53 billets sur le jeu de test.
* Le Feature Engineering axé sur le comportement temporel (demande glissante sur 7j, ratio de prix selon le jour) a structuré la capacité de généralisation du modèle.

**Améliorations possibles**
1. **Correction du Biais d'aversion :** Le modèle étant structurellement pessimiste (biais négatif de sous-estimation), nous devrions modifier la fonction de perte (Loss Function) de CatBoost. Remplacer la MAE standard par une fonction de perte asymétrique (*Quantile Regression* ou *Custom Loss*) pénalisant davantage la sous-estimation permettrait de sécuriser les revenus premium.
2. **Isolation des Lignes Critiques :** Des paires OD complexes comme `rb - bb` méritent l'intégration de métadonnées externes (météo, flux de vacanciers, attractivité le week-end) pour modéliser leurs anomalies.